# Chess Coach Openings RAG

This notebook builds a retrieval augmented generation pipeline for chess opening coaching.

What it does:
- Searches Hugging Face for chess opening datasets and prefers `Lichess/chess-openings` when available.
- Uses opening-level chunks, which fit this dataset because each row is already one ECO/opening/variation record.
- Builds a local TF-IDF vector index instead of calling an embedding API.
- Retrieves with local vector search, BM25, and chess-specific keyword search.
- Merges retrieval results with reciprocal rank fusion.
- Answers through free OpenRouter chat models only, with a fallback list.
- Keeps user and assistant messages persistent in the active notebook session.

References checked on May 15, 2026:
- Hugging Face dataset: https://huggingface.co/datasets/Lichess/chess-openings
- OpenRouter free models: https://openrouter.ai/collections/free-models
- OpenRouter models API: https://openrouter.ai/docs/api/api-reference/models/get-models
- scikit-learn TF-IDF vectors: https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html

## Environment

A local `.venv` has been created at the project root with the current system `python3`. In VS Code or Jupyter, select the kernel from this environment after installing `ipykernel`.

If you need to register the kernel manually from a terminal:

```bash
source .venv/bin/activate
python -m pip install ipykernel
python -m ipykernel install --prefix .venv --name chess-coach-openings-rag --display-name "Chess Coach Openings RAG"
```

In [ ]:
%pip install -q datasets huggingface_hub pandas numpy scikit-learn rank-bm25 python-dotenv requests ipywidgets tqdm joblib


### Imports

In [ ]:
from __future__ import annotations

import json
import math
import os
import re
import time
from html import escape
from pathlib import Path
from typing import Any

import joblib
import numpy as np
import pandas as pd
import requests
from datasets import Dataset, DatasetDict, load_dataset
from dotenv import load_dotenv
from huggingface_hub import HfApi
from IPython.display import display
from rank_bm25 import BM25Okapi
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from tqdm.auto import tqdm


### Configuration

In [ ]:
load_dotenv(".env")

HF_API_KEY = os.getenv("HF_API_KEY") or os.getenv("HUGGINGFACEHUB_API_TOKEN")
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

CACHE_DIR = Path(".rag_cache")
LOCAL_VECTOR_INDEX_NAME = "tfidf_word_1_3_char_3_6"
TFIDF_WORD_NGRAM_RANGE = (1, 3)
TFIDF_CHAR_NGRAM_RANGE = (3, 6)
RRF_K = 60
RANDOM_SEED = 42


### Validate Configuration

In [ ]:
missing = [
    name
    for name, value in {
        "HF_API_KEY": HF_API_KEY,
        "OPENROUTER_API_KEY": OPENROUTER_API_KEY,
    }.items()
    if not value
]
if missing:
    raise RuntimeError(f"Missing required keys in .env: {', '.join(missing)}")

CACHE_DIR.mkdir(exist_ok=True)
np.random.seed(RANDOM_SEED)
print("Configuration loaded. Secrets are present but not displayed.")


## Search Hugging Face For Chess Opening Data

The notebook searches Hugging Face first, then selects the best fit. `Lichess/chess-openings` is preferred because it is compact, CC0 licensed, and already has structured opening names, ECO codes, PGN, UCI, and EPD/FEN-like position data.

### Dataset Search Helpers

In [ ]:
def _safe_get(obj: Any, name: str, default: Any = None) -> Any:
    return getattr(obj, name, default)


def search_chess_opening_datasets(query: str = "chess openings", limit: int = 20) -> pd.DataFrame:
    api = HfApi(token=HF_API_KEY)
    results = list(api.list_datasets(search=query, limit=limit))
    rows = []
    for item in results:
        dataset_id = _safe_get(item, "id", "")
        tags = _safe_get(item, "tags", []) or []
        tags_text = " ".join(tags).lower()
        downloads = _safe_get(item, "downloads", 0) or 0
        likes = _safe_get(item, "likes", 0) or 0
        score = math.log1p(downloads) + math.log1p(likes)
        if dataset_id == "Lichess/chess-openings":
            score += 100
        if "chess" in tags_text:
            score += 5
        if "open" in dataset_id.lower():
            score += 2
        rows.append(
            {
                "id": dataset_id,
                "downloads": downloads,
                "likes": likes,
                "tags": ", ".join(tags[:8]),
                "selection_score": score,
            }
        )
    return pd.DataFrame(rows).sort_values("selection_score", ascending=False).reset_index(drop=True)


### Search Candidate Datasets

In [ ]:
dataset_candidates = search_chess_opening_datasets()
display(dataset_candidates.head(10))


### Select Dataset

In [ ]:
PREFERRED_DATASET_ID = "Lichess/chess-openings"

if PREFERRED_DATASET_ID in set(dataset_candidates["id"]):
    DATASET_ID = PREFERRED_DATASET_ID
elif len(dataset_candidates):
    DATASET_ID = dataset_candidates.iloc[0]["id"]
else:
    DATASET_ID = PREFERRED_DATASET_ID

print(f"Selected dataset: {DATASET_ID}")


### Dataset Loading Helpers

In [ ]:
TEXT_COLUMNS = ["eco-volume", "eco", "name", "pgn", "uci", "epd"]
PARQUET_URL = "hf://datasets/Lichess/chess-openings/data/train-00000-of-00001.parquet"
IMAGE_COLUMNS = {"img", "image", "images"}
VIEWER_ROWS_URL = "https://datasets-server.huggingface.co/rows"


def load_lichess_openings_with_pandas() -> pd.DataFrame:
    """Fast path from the dataset card: read only text columns from the HF parquet file."""
    if DATASET_ID != "Lichess/chess-openings":
        raise ValueError("The direct parquet path is only known for Lichess/chess-openings.")
    return pd.read_parquet(
        PARQUET_URL,
        columns=TEXT_COLUMNS,
        storage_options={"token": HF_API_KEY} if HF_API_KEY else None,
    )


def drop_image_columns(row: dict[str, Any]) -> dict[str, Any]:
    return {key: value for key, value in row.items() if key.lower() not in IMAGE_COLUMNS}


def request_hf_rows_with_retries(params: dict[str, Any], headers: dict[str, str], retries: int = 5) -> dict[str, Any]:
    for attempt in range(retries):
        response = requests.get(VIEWER_ROWS_URL, headers=headers, params=params, timeout=60)
        if response.status_code not in {429, 500, 502, 503, 504}:
            response.raise_for_status()
            return response.json()
        wait_seconds = min(2 ** attempt, 16)
        print(f"HF rows request returned {response.status_code}; retrying in {wait_seconds}s...")
        time.sleep(wait_seconds)
    response.raise_for_status()
    return response.json()


def load_dataset_rows_via_viewer(
    dataset_id: str,
    config: str = "default",
    split: str = "train",
    batch_size: int = 50,
    max_rows: int | None = None,
) -> pd.DataFrame:
    """Load rows through the HF Dataset Viewer API without downloading image parquet payloads."""
    headers = {"Authorization": f"Bearer {HF_API_KEY}"} if HF_API_KEY else {}
    rows: list[dict[str, Any]] = []
    offset = 0
    total_rows = None
    progress = None

    while True:
        length = batch_size if max_rows is None else min(batch_size, max_rows - len(rows))
        if length <= 0:
            break

        payload = request_hf_rows_with_retries(
            params={
                "dataset": dataset_id,
                "config": config,
                "split": split,
                "offset": offset,
                "length": length,
            },
            headers=headers,
        )
        batch = [drop_image_columns(item.get("row", item)) for item in payload.get("rows", [])]

        if total_rows is None:
            total_rows = payload.get("num_rows_total")
            progress_total = min(total_rows, max_rows) if total_rows and max_rows else total_rows
            progress = tqdm(total=progress_total, desc="Requesting HF rows")

        if not batch:
            break

        rows.extend(batch)
        offset += len(batch)
        if progress:
            progress.update(len(batch))
        if total_rows and offset >= total_rows:
            break

    if progress:
        progress.close()
    return pd.DataFrame(rows)


def load_dataset_rows_streaming(dataset_id: str, split: str = "train", max_rows: int | None = None) -> pd.DataFrame:
    """Fallback loader. Streaming avoids a full local download, then drops image fields per row."""
    iterable = load_dataset(dataset_id, split=split, streaming=True, token=HF_API_KEY)
    rows = []
    for idx, row in enumerate(tqdm(iterable, desc="Streaming HF rows")):
        if max_rows is not None and idx >= max_rows:
            break
        rows.append(drop_image_columns(row))
    return pd.DataFrame(rows)


### Load Dataset

In [ ]:
split_name = "train"
try:
    df = load_lichess_openings_with_pandas()
except Exception as exc:
    print(f"Direct parquet column load failed: {exc}")
    print("Falling back to Hugging Face Dataset Viewer API.")
    try:
        df = load_dataset_rows_via_viewer(DATASET_ID, config="default", split=split_name, batch_size=50)
    except Exception as viewer_exc:
        print(f"Dataset Viewer API load failed: {viewer_exc}")
        print("Falling back to Hugging Face streaming mode.")
        df = load_dataset_rows_streaming(DATASET_ID, split=split_name)

print(f"Loaded {len(df):,} text rows from split '{split_name}' without keeping image columns.")
display(df.head())


## Chunking Strategy

For chess openings, the cleanest chunk is usually one opening variation per row. A single Lichess row contains the opening name, ECO code, move order, UCI moves, and position. Splitting inside that record would separate the name from the line and hurt retrieval. The helper below still has a fallback text splitter for larger datasets with verbose move explanations.

### Chunking Helpers

In [ ]:
def first_present(row: pd.Series, names: list[str], default: str = "") -> str:
    for name in names:
        if name in row and pd.notna(row[name]) and str(row[name]).strip():
            return str(row[name]).strip()
    return default


def row_to_document(row: pd.Series, row_id: int) -> dict[str, Any]:
    opening_name = first_present(row, ["name", "Opening_type", "opening", "title"])
    eco = first_present(row, ["eco", "ECO", "eco_code"])
    eco_volume = first_present(row, ["eco-volume", "eco_volume", "volume"])
    pgn = first_present(row, ["pgn", "PGN", "moves", "line"])
    uci = first_present(row, ["uci", "UCI"])
    epd = first_present(row, ["epd", "fen", "FEN"])
    context = first_present(row, ["Context", "context", "description", "text"])

    parts = []
    if opening_name:
        parts.append(f"Opening: {opening_name}")
    if eco:
        parts.append(f"ECO: {eco}")
    if eco_volume:
        parts.append(f"ECO volume: {eco_volume}")
    if pgn:
        parts.append(f"PGN move order: {pgn}")
    if uci:
        parts.append(f"UCI move order: {uci}")
    if epd:
        parts.append(f"Position EPD/FEN: {epd}")
    if context:
        parts.append(f"Dataset context: {context}")

    if not parts:
        parts = [json.dumps(row.dropna().to_dict(), ensure_ascii=True)]

    return {
        "doc_id": f"{DATASET_ID}:{row_id}",
        "opening_name": opening_name,
        "eco": eco,
        "pgn": pgn,
        "uci": uci,
        "epd": epd,
        "text": "\n".join(parts),
        "source_dataset": DATASET_ID,
    }


def split_long_text(text: str, max_chars: int = 1400, overlap: int = 180) -> list[str]:
    if len(text) <= max_chars:
        return [text]
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + max_chars, len(text))
        boundary = max(text.rfind("\n", start, end), text.rfind(". ", start, end))
        if boundary > start + max_chars // 2:
            end = boundary + 1
        chunks.append(text[start:end].strip())
        if end >= len(text):
            break
        start = max(0, end - overlap)
    return [chunk for chunk in chunks if chunk]


### Build Retrieval Documents

In [ ]:
documents = []
for row_id, row in df.iterrows():
    base_doc = row_to_document(row, row_id)
    for chunk_id, chunk in enumerate(split_long_text(base_doc["text"])):
        doc = dict(base_doc)
        doc["chunk_id"] = chunk_id
        doc["text"] = chunk
        documents.append(doc)

docs_df = pd.DataFrame(documents)
print(f"Prepared {len(docs_df):,} retrieval chunks.")
display(docs_df.head())


## Build Retrieval Indexes

This notebook uses a local TF-IDF vector index instead of remote embeddings. For chess openings, that is a better default because the important evidence is often exact or near-exact text: ECO codes, opening names, SAN/PGN moves, UCI moves, and FEN/EPD fragments. The index is cached in `.rag_cache/` and rebuilds only when the dataset or vector settings change.


### Text Search Helpers

In [ ]:
TOKEN_PATTERN = re.compile(r"[a-zA-Z0-9+#=O\-]+")


def normalize_text(text: str) -> str:
    text = str(text).lower()
    text = text.replace("0-0", "O-O").replace("0-0-0", "O-O-O")
    return re.sub(r"\s+", " ", text).strip()


def tokenize(text: str) -> list[str]:
    return TOKEN_PATTERN.findall(normalize_text(text))


### Build BM25 Index

In [ ]:
docs_df["search_text"] = docs_df["text"].map(normalize_text)
tokenized_corpus = docs_df["search_text"].map(tokenize).tolist()
bm25 = BM25Okapi(tokenized_corpus)

print("BM25 and keyword corpus ready.")


### Vector Index Helpers

In [ ]:
def safe_slug(value: str) -> str:
    return re.sub(r"[^a-zA-Z0-9_.-]+", "_", value).strip("_")


def build_local_vector_index(texts: list[str]) -> dict[str, Any]:
    word_vectorizer = TfidfVectorizer(
        token_pattern=TOKEN_PATTERN.pattern,
        ngram_range=TFIDF_WORD_NGRAM_RANGE,
        lowercase=True,
        sublinear_tf=True,
        dtype=np.float32,
    )
    char_vectorizer = TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=TFIDF_CHAR_NGRAM_RANGE,
        sublinear_tf=True,
        dtype=np.float32,
    )

    word_matrix = word_vectorizer.fit_transform(texts)
    char_matrix = char_vectorizer.fit_transform(texts)
    doc_vectors = normalize(hstack([word_matrix, char_matrix], format="csr"), copy=False)
    return {
        "word_vectorizer": word_vectorizer,
        "char_vectorizer": char_vectorizer,
        "doc_vectors": doc_vectors,
    }


### Build Or Load Vector Index

In [ ]:
vector_cache_path = CACHE_DIR / (
    f"vectors_{safe_slug(DATASET_ID)}_{safe_slug(LOCAL_VECTOR_INDEX_NAME)}_"
    f"n{len(docs_df)}.joblib"
)

if vector_cache_path.exists():
    local_vector_index = joblib.load(vector_cache_path)
    print(f"Loaded cached local vector index from {vector_cache_path}")
else:
    local_vector_index = build_local_vector_index(docs_df["search_text"].tolist())
    joblib.dump(local_vector_index, vector_cache_path)
    print(f"Saved local vector index to {vector_cache_path}")

doc_vectors = local_vector_index["doc_vectors"]
print(doc_vectors.shape)


## Reciprocal Rank Fusion Retrieval

### Retrieval Helpers

In [ ]:
query_vector_cache: dict[str, Any] = {}


def vectorize_query(query: str):
    cache_key = normalize_text(query)
    if cache_key not in query_vector_cache:
        word_vector = local_vector_index["word_vectorizer"].transform([cache_key])
        char_vector = local_vector_index["char_vectorizer"].transform([cache_key])
        query_vector_cache[cache_key] = normalize(
            hstack([word_vector, char_vector], format="csr"),
            copy=False,
        )
    return query_vector_cache[cache_key]


def top_indices(scores: np.ndarray, top_n: int) -> list[int]:
    top_n = min(top_n, len(scores))
    if top_n <= 0:
        return []
    candidate_idx = np.argpartition(-scores, top_n - 1)[:top_n]
    return candidate_idx[np.argsort(-scores[candidate_idx])].tolist()


def vector_ranking(query: str, pool_size: int = 80) -> tuple[list[int], np.ndarray]:
    q = vectorize_query(query)
    scores = np.asarray((doc_vectors @ q.T).todense()).reshape(-1)
    return top_indices(scores, pool_size), scores


def bm25_ranking(query: str, pool_size: int = 80) -> tuple[list[int], np.ndarray]:
    scores = np.asarray(bm25.get_scores(tokenize(query)), dtype=np.float32)
    return top_indices(scores, pool_size), scores


def keyword_scores(query: str) -> np.ndarray:
    q_norm = normalize_text(query)
    q_tokens = set(tokenize(query))
    scores = np.zeros(len(docs_df), dtype=np.float32)
    for i, row in docs_df.iterrows():
        text = row["search_text"]
        name = normalize_text(row.get("opening_name", ""))
        eco = normalize_text(row.get("eco", ""))
        pgn = normalize_text(row.get("pgn", ""))
        if q_norm and q_norm in name:
            scores[i] += 15
        if q_norm and q_norm in text:
            scores[i] += 5
        if eco and eco in q_norm:
            scores[i] += 10
        if pgn and pgn in q_norm:
            scores[i] += 8
        doc_tokens = set(tokenized_corpus[i])
        scores[i] += len(q_tokens & doc_tokens)
    return scores


def keyword_ranking(query: str, pool_size: int = 80) -> tuple[list[int], np.ndarray]:
    scores = keyword_scores(query)
    return top_indices(scores, pool_size), scores


def reciprocal_rank_fusion(
    rankings: dict[str, list[int]],
    weights: dict[str, float] | None = None,
    k: int = RRF_K,
) -> list[tuple[int, float]]:
    weights = weights or {method: 1.0 for method in rankings}
    fused: dict[int, float] = {}
    for method, ranked_indices in rankings.items():
        weight = weights.get(method, 1.0)
        for rank, doc_idx in enumerate(ranked_indices, start=1):
            fused[doc_idx] = fused.get(doc_idx, 0.0) + weight / (k + rank)
    return sorted(fused.items(), key=lambda item: item[1], reverse=True)


def retrieve(query: str, top_k: int = 8, pool_size: int = 80) -> pd.DataFrame:
    vec_rank, vec_scores = vector_ranking(query, pool_size=pool_size)
    bm_rank, bm_scores = bm25_ranking(query, pool_size=pool_size)
    kw_rank, kw_scores = keyword_ranking(query, pool_size=pool_size)

    rankings = {"vector": vec_rank, "bm25": bm_rank, "keyword": kw_rank}
    fused = reciprocal_rank_fusion(
        rankings,
        weights={"vector": 1.15, "bm25": 1.0, "keyword": 1.25},
    )[:top_k]

    rows = []
    for doc_idx, fused_score in fused:
        row = docs_df.iloc[doc_idx].to_dict()
        row.update(
            {
                "doc_index": doc_idx,
                "rrf_score": fused_score,
                "vector_score": float(vec_scores[doc_idx]),
                "bm25_score": float(bm_scores[doc_idx]),
                "keyword_score": float(kw_scores[doc_idx]),
                "vector_rank": vec_rank.index(doc_idx) + 1 if doc_idx in vec_rank else None,
                "bm25_rank": bm_rank.index(doc_idx) + 1 if doc_idx in bm_rank else None,
                "keyword_rank": kw_rank.index(doc_idx) + 1 if doc_idx in kw_rank else None,
            }
        )
        rows.append(row)
    return pd.DataFrame(rows)


### Retrieval Preview

In [ ]:
retrieval_preview = retrieve("What should I know about the Sicilian Defense Najdorf?", top_k=5)
display(retrieval_preview[["opening_name", "eco", "pgn", "rrf_score", "vector_rank", "bm25_rank", "keyword_rank"]])


## Free OpenRouter Model Selection

The notebook asks the OpenRouter models API for the current model list, filters to free text-output models, and then uses a preference order for chess coaching. The fallback `openrouter/free` route is kept last so the notebook can still work when specific free models change.

### OpenRouter Model Helpers

In [ ]:
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

PREFERRED_FREE_MODEL_IDS = [
    "openrouter/owl-alpha",
    "deepseek/deepseek-v4-flash:free",
    "google/gemma-4-31b-it:free",
    "openai/gpt-oss-20b:free",
    "qwen/qwen3-coder:free",
    "meta-llama/llama-3.3-70b-instruct:free",
    "openrouter/free",
]


def _price_is_zero(value: Any) -> bool:
    try:
        return float(value) == 0.0
    except (TypeError, ValueError):
        return False


def list_free_openrouter_models() -> list[dict[str, Any]]:
    headers = {"Authorization": f"Bearer {OPENROUTER_API_KEY}"}
    response = requests.get(
        f"{OPENROUTER_BASE_URL}/models",
        headers=headers,
        params={"output_modalities": "text"},
        timeout=30,
    )
    response.raise_for_status()
    models = response.json().get("data", [])
    free_models = []
    for model in models:
        model_id = model.get("id", "")
        name = model.get("name", "")
        architecture = model.get("architecture") or {}
        pricing = model.get("pricing") or {}
        output_modalities = architecture.get("output_modalities") or []
        is_text_output = not output_modalities or "text" in output_modalities
        is_free = (
            model_id.endswith(":free")
            or "free" in name.lower()
            or (
                _price_is_zero(pricing.get("prompt"))
                and _price_is_zero(pricing.get("completion"))
                and _price_is_zero(pricing.get("request", 0))
            )
        )
        if is_text_output and is_free:
            free_models.append(model)
    return free_models


### Select Generation Models

In [ ]:
try:
    free_models = list_free_openrouter_models()
except Exception as exc:
    print(f"Could not fetch live OpenRouter models: {exc}")
    free_models = []

free_model_ids = {model.get("id") for model in free_models}
GENERATION_MODELS = [model_id for model_id in PREFERRED_FREE_MODEL_IDS if model_id in free_model_ids]

if "openrouter/free" not in GENERATION_MODELS:
    GENERATION_MODELS.append("openrouter/free")

print("Free OpenRouter models selected for fallback order:")
for model_id in GENERATION_MODELS:
    print("-", model_id)

free_models_preview = pd.DataFrame(
    [
        {
            "id": model.get("id"),
            "name": model.get("name"),
            "context_length": model.get("context_length"),
        }
        for model in free_models
    ]
)
display(free_models_preview.head(20))


## Answer Generation

The next helper cell owns display formatting. The following core cell owns retrieval, model calling, session memory, and the `ask()` entry point.


### Answer Display Helpers


In [ ]:
from IPython.display import display
from typing import Any


def plain_text_response(text: str) -> str:
    """Remove common Markdown artifacts when a free model ignores the plain-text instruction."""
    text = text.strip()
    text = re.sub(r"```(?:\w+)?\n?(.*?)```", r"\1", text, flags=re.S)
    text = re.sub(r"^\s*#{1,6}\s+", "", text, flags=re.M)
    text = re.sub(r"^\s*[*+]\s+", "- ", text, flags=re.M)
    text = re.sub(r"^\s*>\s?", "", text, flags=re.M)
    text = re.sub(r"\*\*(.*?)\*\*", r"\1", text)
    text = re.sub(r"__(.*?)__", r"\1", text)
    text = re.sub(r"(?<!\*)\*([^*\n]+)\*(?!\*)", r"\1", text)
    text = re.sub(r"(?<!_)_([^_\n]+)_(?!_)", r"\1", text)
    text = re.sub(r"`([^`]*)`", r"\1", text)
    text = re.sub(r"\[([^\]]+)\]\([^)]+\)", r"\1", text)
    text = re.sub(r"^\s*\|?\s*:?-{3,}:?\s*(\|\s*:?-{3,}:?\s*)+\|?\s*$", "", text, flags=re.M)
    text = re.sub(r"^\|(.+)\|$", lambda match: "  ".join(part.strip() for part in match.group(1).split("|")), text, flags=re.M)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def compact_source_value(value: Any, digits: int | None = None) -> str:
    if pd.isna(value):
        return "-"
    if digits is not None:
        return f"{float(value):.{digits}f}"
    if isinstance(value, float) and value.is_integer():
        return str(int(value))
    return str(value)


def print_answer_text(question: str, answer: str, model_used: str) -> None:
    print("Chess Opening Coach")
    print(f"Model: {model_used}")
    print("=" * 72)
    print(f"Question: {question}\n")
    print(answer)


def format_sources_text(results: pd.DataFrame) -> str:
    lines = [
        "Retrieved sources",
        "Fusion score plus vector, BM25, and keyword ranks",
        "-" * 72,
    ]
    for source_rank, (_, row) in enumerate(results.iterrows(), start=1):
        score = compact_source_value(row.get("rrf_score"), digits=4)
        opening = compact_source_value(row.get("opening_name"))
        eco = compact_source_value(row.get("eco"))
        pgn = compact_source_value(row.get("pgn"))
        vector_rank = compact_source_value(row.get("vector_rank"))
        bm25_rank = compact_source_value(row.get("bm25_rank"))
        keyword_rank = compact_source_value(row.get("keyword_rank"))
        lines.extend(
            [
                f"{source_rank}. {opening} ({eco})",
                f"   PGN: {pgn}",
                f"   Score: {score} | ranks: vector {vector_rank}, BM25 {bm25_rank}, keyword {keyword_rank}",
            ]
        )
    return "\n".join(lines)


### Core Answer Logic

In [ ]:
SYSTEM_PROMPT = """
You are a practical chess opening coach. Answer from the retrieved opening records first.
Return plain text only. Do not use Markdown, bullet markers, tables, code fences, headings,
bold text, italic text, or inline code formatting. Use short labeled lines like Opening,
Main line, White plan, Black plan, Risks, and Recommendation. Give concrete move orders,
ECO codes when available, plans for both sides, typical risks, and one or two beginner-friendly
recommendations. If the retrieved data is insufficient, say what is missing instead of inventing a line.
""".strip()

SESSION_MESSAGES: list[dict[str, str]] = []


def format_context(results: pd.DataFrame) -> str:
    blocks = []
    for rank, (_, row) in enumerate(results.iterrows(), start=1):
        blocks.append(
            "\n".join(
                [
                    f"[Source {rank}]",
                    f"Opening: {row.get('opening_name', '')}",
                    f"ECO: {row.get('eco', '')}",
                    f"PGN: {row.get('pgn', '')}",
                    f"UCI: {row.get('uci', '')}",
                    f"EPD/FEN: {row.get('epd', '')}",
                    f"Text: {row.get('text', '')}",
                ]
            )
        )
    return "\n\n".join(blocks)


def call_openrouter_chat(messages: list[dict[str, str]], models: list[str] | None = None) -> tuple[str, str]:
    models = models or GENERATION_MODELS
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
        "HTTP-Referer": "http://localhost",
        "X-Title": "Chess Coach Openings RAG",
    }
    last_error = None
    for model_id in models:
        payload = {
            "model": model_id,
            "messages": messages,
            "temperature": 0.35,
            "max_tokens": 900,
        }
        try:
            response = requests.post(
                f"{OPENROUTER_BASE_URL}/chat/completions",
                headers=headers,
                json=payload,
                timeout=90,
            )
            if response.status_code in {429, 500, 502, 503, 504}:
                last_error = RuntimeError(f"{model_id}: {response.status_code} {response.text[:300]}")
                continue
            response.raise_for_status()
            data = response.json()
            answer = data["choices"][0]["message"]["content"]
            return answer, model_id
        except Exception as exc:
            last_error = exc
            continue
    raise RuntimeError(f"All OpenRouter free model calls failed. Last error: {last_error}")


def ask(question: str, top_k: int = 8, show_sources: bool = True, verbose: bool = True) -> str:
    results = retrieve(question, top_k=top_k)
    context = format_context(results)
    recent_history = SESSION_MESSAGES[-8:]
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        *recent_history,
        {
            "role": "user",
            "content": (
                "Retrieved chess opening records:\n"
                f"{context}\n\n"
                f"Question: {question}"
            ),
        },
    ]
    answer, model_used = call_openrouter_chat(messages)
    answer = plain_text_response(answer)
    SESSION_MESSAGES.append({"role": "user", "content": question})
    SESSION_MESSAGES.append({"role": "assistant", "content": answer})

    if verbose:
        print_answer_text(question, answer, model_used)
    if show_sources:
        print()
        print(format_sources_text(results))
    return answer


def clear_chat_history(verbose: bool = True) -> None:
    SESSION_MESSAGES.clear()
    if verbose:
        print("Session chat history cleared.")


In [ ]:
# Example question. Edit and rerun.
_ = ask("tell me about the english opening", top_k=8)


## Notebook Chatbot UI

Run the next cells for a session-only chatbot inside the notebook. Messages stay in `SESSION_MESSAGES` until you restart the kernel, click Clear, or call `clear_chat_history()`.

### Chat Display Helpers


In [ ]:
import ipywidgets as widgets
from html import escape
from IPython.display import display
from typing import Any


CHAT_HEADER_HTML = """
<style>
  .coach-chat-header { box-sizing:border-box; display:flex; align-items:center; justify-content:space-between; gap:16px; padding:16px 18px; background:#111827; border-radius:8px 8px 0 0; border-bottom:1px solid #2f3746; font-family:system-ui, -apple-system, Segoe UI, sans-serif; }
  .coach-chat-title { display:flex; align-items:center; gap:12px; min-width:0; }
  .coach-chat-mark { width:42px; height:42px; flex:0 0 42px; display:grid; place-items:center; color:#111827; background:#f7d774; border:1px solid #e7c24d; border-radius:8px; font-weight:900; font-size:24px; }
  .coach-chat-name { color:#f9fafb; font-size:20px; font-weight:800; letter-spacing:0; line-height:1.2; }
  .coach-chat-subtitle { color:#cbd5e1; font-size:13px; line-height:1.35; margin-top:3px; }
  .coach-chat-badges { display:flex; flex-wrap:wrap; justify-content:flex-end; gap:8px; }
  .coach-chat-badge { color:#d9f99d; background:#1f2937; border:1px solid #3f4b5f; padding:5px 8px; border-radius:7px; font-size:12px; font-weight:700; }
  .coach-bubble-row { display:flex; gap:9px; margin:12px 0; align-items:flex-end; }
  .coach-bubble-row.user { flex-direction:row-reverse; }
  .coach-avatar { width:30px; height:30px; flex:0 0 30px; display:grid; place-items:center; border-radius:8px; font-size:12px; font-weight:900; font-family:system-ui, -apple-system, Segoe UI, sans-serif; }
  .coach-avatar.coach { background:#f7d774; color:#111827; border:1px solid #e7c24d; }
  .coach-avatar.user { background:#2563eb; color:#fff; border:1px solid #1d4ed8; }
  .coach-bubble { box-sizing:border-box; max-width:78%; padding:10px 12px; border-radius:8px; line-height:1.5; white-space:pre-wrap; overflow-wrap:anywhere; box-shadow:0 8px 22px rgba(15,23,42,0.08); font-family:system-ui, -apple-system, Segoe UI, sans-serif; font-size:14px; }
  .coach-bubble.coach { color:#111827; background:#ffffff; border:1px solid #d7dce4; }
  .coach-bubble.user { color:#fff; background:#1d4ed8; border:1px solid #1e40af; }
  .coach-label { font-size:11px; line-height:1; text-transform:uppercase; font-weight:800; letter-spacing:0; margin-bottom:6px; opacity:.72; }
  .coach-chip { display:inline-flex; align-items:center; min-height:30px; border:1px solid #d7dce4; border-radius:7px; padding:0 10px; color:#344054; background:#fff; font-size:12px; font-weight:700; font-family:system-ui, -apple-system, Segoe UI, sans-serif; }
  @media (max-width: 720px) { .coach-chat-header { align-items:flex-start; flex-direction:column; } .coach-chat-badges { justify-content:flex-start; } .coach-bubble { max-width:88%; } }
</style>
<div class="coach-chat-header">
  <div class="coach-chat-title">
    <div class="coach-chat-mark">N</div>
    <div>
      <div class="coach-chat-name">Chess Opening Coach</div>
      <div class="coach-chat-subtitle">Ask for plans, move orders, traps, and practical recommendations.</div>
    </div>
  </div>
  <div class="coach-chat-badges">
    <span class="coach-chat-badge">RAG</span>
    <span class="coach-chat-badge">Openings</span>
    <span class="coach-chat-badge">Session memory</span>
  </div>
</div>
"""


def chat_bubble(role: str, text: str) -> widgets.HTML:
    is_user = role == "user"
    align = "flex-end" if is_user else "flex-start"
    label = "You" if is_user else "Coach"
    avatar_class = "user" if is_user else "coach"
    bubble_class = "user" if is_user else "coach"
    avatar = "Y" if is_user else "C"
    return widgets.HTML(
        value=f"""
        <div class="coach-bubble-row {bubble_class}" style="justify-content:{align};">
          <div class="coach-avatar {avatar_class}">{avatar}</div>
          <div class="coach-bubble {bubble_class}">
            <div class="coach-label">{label}</div>
            <div>{escape(text).replace(chr(10), '<br>')}</div>
          </div>
        </div>
        """
    )


### Chat Logic Helper

In [ ]:
def launch_chatbot() -> None:
    question_box = widgets.Text(
        description="",
        placeholder="Ask about the Ruy Lopez, Sicilian traps, English move orders...",
        continuous_update=False,
        layout=widgets.Layout(width="100%", height="38px"),
    )
    send_button = widgets.Button(description="Ask", icon="paper-plane", tooltip="Send question", layout=widgets.Layout(width="96px", height="38px"))
    send_button.style.button_color = "#1d4ed8"
    send_button.style.text_color = "#ffffff"
    send_button.style.font_weight = "700"
    clear_button = widgets.Button(description="Clear", icon="trash", tooltip="Clear chat history", layout=widgets.Layout(width="96px", height="34px"))
    clear_button.style.button_color = "#ffffff"
    clear_button.style.font_weight = "700"
    status = widgets.HTML(value="<span class='coach-chip'>Ready</span>")
    messages_box = widgets.VBox(
        children=(chat_bubble("assistant", "Ready. Give me an opening, move order, or plan you want to understand."),),
        layout=widgets.Layout(
            width="100%",
            min_height="340px",
            max_height="560px",
            overflow_y="auto",
            padding="16px 18px",
            border_left="1px solid #d7dce4",
            border_right="1px solid #d7dce4",
            background_color="#f6f4ee",
        )
    )
    starter_prompts = [
        "Give me a simple Ruy Lopez plan for White",
        "What should Black watch for in the Sicilian?",
        "Explain the English Opening move order",
    ]
    prompt_buttons = [
        widgets.Button(description=prompt, tooltip=prompt, layout=widgets.Layout(min_width="220px", height="32px"))
        for prompt in starter_prompts
    ]
    for prompt_button in prompt_buttons:
        prompt_button.style.button_color = "#ffffff"
        prompt_button.style.font_weight = "600"
    starter_row = widgets.HBox(
        prompt_buttons,
        layout=widgets.Layout(width="100%", gap="8px", padding="10px 14px", flex_flow="row wrap", border_left="1px solid #d7dce4", border_right="1px solid #d7dce4", background_color="#f9fafb"),
    )
    input_row = widgets.HBox(
        [question_box, send_button],
        layout=widgets.Layout(width="100%", gap="10px", padding="12px 14px", align_items="center", border_left="1px solid #d7dce4", border_right="1px solid #d7dce4", background_color="#ffffff"),
    )

    def append_message(role: str, text: str) -> None:
        messages_box.children = (*messages_box.children, chat_bubble(role, text))

    def send_message(_event: Any = None) -> None:
        question = question_box.value.strip()
        if not question or send_button.disabled:
            return
        question_box.value = ""
        append_message("user", question)
        send_button.disabled = True
        question_box.disabled = True
        for prompt_button in prompt_buttons:
            prompt_button.disabled = True
        status.value = "<span class='coach-chip'>Thinking through candidate lines...</span>"
        try:
            answer = ask(question, top_k=8, show_sources=False, verbose=False)
            append_message("assistant", answer)
        except Exception as exc:
            append_message("assistant", f"Error: {exc}")
        finally:
            status.value = "<span class='coach-chip'>Ready</span>"
            send_button.disabled = False
            question_box.disabled = False
            for prompt_button in prompt_buttons:
                prompt_button.disabled = False
            focus_input = getattr(question_box, "focus", None)
            if callable(focus_input):
                focus_input()

    def use_starter(prompt: str) -> None:
        question_box.value = prompt
        send_message()

    def on_clear(_button: widgets.Button) -> None:
        clear_chat_history(verbose=False)
        messages_box.children = (chat_bubble("assistant", "Ready. Give me an opening, move order, or plan you want to understand."),)
        status.value = "<span class='coach-chip'>Ready</span>"
        question_box.value = ""

    submit_handler = getattr(question_box, "on_submit", None)
    if callable(submit_handler):
        submit_handler(send_message)
    for prompt, prompt_button in zip(starter_prompts, prompt_buttons):
        prompt_button.on_click(lambda _button, prompt=prompt: use_starter(prompt))
    send_button.on_click(send_message)
    clear_button.on_click(on_clear)
    controls = widgets.HBox(
        [status, clear_button],
        layout=widgets.Layout(width="100%", align_items="center", justify_content="space-between", padding="10px 14px", border_left="1px solid #d7dce4", border_right="1px solid #d7dce4", border_bottom="1px solid #d7dce4", background_color="#f9fafb"),
    )
    display(widgets.VBox([widgets.HTML(CHAT_HEADER_HTML), messages_box, starter_row, input_row, controls], layout=widgets.Layout(width="100%", max_width="980px")))


### Launch Chatbot

In [ ]:
launch_chatbot()
